**Crime and Enrichment Data Pipeline**

Package Imports

In [ ]:
import pandas as pd
import os

Ingestion Layer (Crime Data)

In [ ]:
basePath = '6YearSpanDatasets'

# This ingestion method loads each individual force's monthly data one by one, appending it into an array and then adding it to a dataframe

# List of forces that are going to be analysed
forces = ['metropolitan', 'thames', 'surrey', 'sussex']

# Array which is cumulatively appended for each force's crime data for each month between January 2019 to December 2024
allData = []

# Checks each folder in the updatedDatasets folder
for folder in os.listdir(basePath):
    # Concatenates the folder name to the updatedDatasets to create a new file path
    filePath = os.path.join(basePath, folder)
    # Checks if the folder actually exists
    if os.path.isdir(filePath):
        # Checks each file in the file path
        for file in os.listdir(filePath):
            # Checks the file contains one of the police forces in the filename
            if any(force in file.lower() for force in forces):
                # Updates file path to include the folder
               file_path = os.path.join(filePath, file)
               
               df=pd.read_csv(file_path)
               df['Month'] = folder
               allData.append(df)
    
# If statement to check the ingestion layer correctly loaded data
if allData:
    masterDf = pd.concat(allData, ignore_index=True)
else:
    print("No data found to process")


Initial Data Inspection

In [ ]:
# Shows number of records in the master data
print(f"Before preprocessing, there are {len(masterDf)} values in the data")

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

# Shows the unique values for each column
for col in masterDf:
    print(f" - The unique values for {col} are {masterDf[col].unique()}")


Deduplication

In [ ]:
# Checks the number of total records in the crime data before any transformation or cleaning
totalBeforeDupRemoval = len(masterDf)
print(f"There are {totalBeforeDupRemoval} values in the dataset before any processing at all")

# Checks the number of duplicate records in the crime data
duplicateCount = masterDf.duplicated().sum()
print(f"There are {duplicateCount} duplicate rows.")

# Drops the duplicate records
masterDf = masterDf.drop_duplicates()

#
totalAfterDupRemoval = len(masterDf)
print(f"There are {totalAfterDupRemoval} values in the dataset after removing duplicates")

Null Value Handling

In [ ]:
# From cell 4 there are missing values in Crime ID, Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category and Context

# Shows each column and if they have any null values
print(f"\nList of null values in every column before handling null values\n {masterDf.isnull().sum()}\n")

# In the data, Crime ID, Longitude, Latitude, LSOA Code, LSOA Name, Last outcome category and Context contained null values

# Context was empty for every single record so can be dropped
masterDf = masterDf.drop("Context", axis='columns')

# For remaining columns, too much crime data would be lost if dropped so placeholders were assigned instead

# Crime ID: add placeholder value "No Crime ID"
# Longitude and Latitude: add placeholder value 0
# LSOA code: add placeholder value "No LSOA Code Assigned"
# LSOA name: add placeholder value "No LSOA name Assigned"
# Last outcome category: add placeholder value "Unknown outcome category"
# Context: drop since every record has null values in context

placeholders = {
    'Crime ID': 'No Crime ID',
    'Longitude': 0,
    'Latitude' : 0,
    'LSOA code' : 'No LSOA Code assigned',
    'LSOA name' : 'No LSOA Name assigned',
    'Last outcome category' : 'Unknown outcome category'
}

masterDf = masterDf.fillna(value=placeholders)

# Shows each column and if they have any null values
print(f"\nList of null values in every column \n {masterDf.isnull().sum()}\n")

Column Handling

In [ ]:
# Dropping "Falls within" column as each record mirrored the value within the "Reported by" column
masterDf = masterDf.drop("Falls within", axis='columns')

Text Standardisation

In [ ]:
# Clean Force names to be just the name of the police force without "Force" or other keywords
masterDf['Force'] = masterDf['Reported by'].str.replace("Police", "").str.strip()

# Clean metropolitan
masterDf.loc[masterDf['Force'].str.contains('Metropolitan', case=False), 'Force'] = 'Metropolitan'
masterDf['Force'] = masterDf['Force'].str.title()

# Standardize 'Crime type' and 'Last outcome category'
masterDf['Crime type'] = masterDf['Crime type'].str.strip().str.capitalize()
masterDf['Last outcome category'] = masterDf['Last outcome category'].str.strip().str.capitalize()

# Dropped Reported by column since a cleaner, standardised Force column was derived from it
masterDf = masterDf.drop('Reported by', axis='columns')

# For loop to ensure that the column was definitely dropped
for col in masterDf:
    print(col)
    
print(f"\n{masterDf.head()}")

Year Conversion 

In [ ]:
# Converting the existing time data into a readable pandas time attribute for future analysis
yearConversion = pd.to_datetime(masterDf['Month'])
masterDf['Year'] = yearConversion.dt.year

Crime Data Aggregation

In [ ]:
# Creating the aggregations, grouping by the defined grain (Force X Year X LSOA name x Crime Type)
crimeAggregation = masterDf.groupby(['Force', 'Year', 'LSOA name', 'Crime type']).agg(
    crimeCount=('Crime ID', 'count'),
    Latitude=('Latitude', 'mean'),
    Longitude=('Longitude', 'mean')
).reset_index()

crimeAggregation = crimeAggregation.rename(columns={'crimeCount': 'Crime count'})

crimeAggregation.head(10)

Ingestion Layer (ONS Population Data)

In [ ]:
xls = pd.ExcelFile('policeforceareas1991to2024.xlsx')
# Extracting only the Mid-2011 to Mid-2020 and Mid-2021 to Mid-2024 sheet as these contain the years that are to be analysed
pop_11_20 = pd.read_excel(xls, 'Mid-2011 to Mid-2020', skiprows=3)
pop_21_24 = pd.read_excel(xls, 'Mid-2021 to Mid-2024', skiprows=3)

# Concatenating the data together to form one raw data set of the ONS Population
popAll = pd.concat([pop_11_20, pop_21_24], axis=0)
popAll.head(12)




Feature Engineering

In [ ]:
# Calculates the total population for a force for a given year
populationColumns = popAll.columns[3:]
popAll['Population'] = popAll[populationColumns].sum(axis=1)

popClean = popAll[['PFA 2023 Name', 'Year', 'Population']].copy()
popClean.columns = ['Force', 'Year', 'Population']
popClean = popClean.reset_index(drop=True)

print(popClean.head())

Text Standardisation

In [ ]:
popClean['Force'] = popClean['Force'].str.replace("Police", "").str.strip()


Population Data Aggregation


In [ ]:
forceYearTotals = masterDf.groupby(['Force', 'Year']).size().reset_index(name='Total Force Crime Count')
forceRatesBaseline = pd.merge(forceYearTotals, popClean, on=['Force', 'Year'], how='left')
forceRatesBaseline['Force Crimes per 1000'] = (forceRatesBaseline['Total Force Crime Count'] / forceRatesBaseline['Population']) * 1000

Ingestion Layer (Index of Deprivation Data)

In [ ]:
# Dictionary for the LSOA names that are covered by the chosen police forces
force_lookup = {
    # Metropolitan 
    'Barking and Dagenham': 'Metropolitan', 'Barnet': 'Metropolitan', 'Bexley': 'Metropolitan',
    'Brent': 'Metropolitan', 'Bromley': 'Metropolitan', 'Camden': 'Metropolitan',
    'Croydon': 'Metropolitan', 'Ealing': 'Metropolitan', 'Enfield': 'Metropolitan',
    'Greenwich': 'Metropolitan', 'Hackney': 'Metropolitan', 'Hammersmith and Fulham': 'Metropolitan',
    'Haringey': 'Metropolitan', 'Harrow': 'Metropolitan', 'Havering': 'Metropolitan',
    'Hillingdon': 'Metropolitan', 'Hounslow': 'Metropolitan', 'Islington': 'Metropolitan',
    'Kensington and Chelsea': 'Metropolitan', 'Kingston upon Thames': 'Metropolitan',
    'Lambeth': 'Metropolitan', 'Lewisham': 'Metropolitan', 'Merton': 'Metropolitan',
    'Newham': 'Metropolitan', 'Redbridge': 'Metropolitan', 'Richmond upon Thames': 'Metropolitan',
    'Southwark': 'Metropolitan', 'Sutton': 'Metropolitan', 'Tower Hamlets': 'Metropolitan',
    'Waltham Forest': 'Metropolitan', 'Wandsworth': 'Metropolitan', 'Westminster': 'Metropolitan',
    'City of London': 'Metropolitan', 
    # Surrey
    'Elmbridge': 'Surrey', 'Epsom and Ewell': 'Surrey', 'Guildford': 'Surrey',
    'Mole Valley': 'Surrey', 'Reigate and Banstead': 'Surrey', 'Runnymede': 'Surrey',
    'Spelthorne': 'Surrey', 'Surrey Heath': 'Surrey', 'Tandridge': 'Surrey',
    'Waverley': 'Surrey', 'Woking': 'Surrey',

    # Sussex
    'Adur': 'Sussex', 'Arun': 'Sussex', 'Brighton and Hove': 'Sussex',
    'Chichester': 'Sussex', 'Crawley': 'Sussex', 'Eastbourne': 'Sussex',
    'Hastings': 'Sussex', 'Horsham': 'Sussex', 'Lewes': 'Sussex',
    'Mid Sussex': 'Sussex', 'Rother': 'Sussex', 'Wealden': 'Sussex', 'Worthing': 'Sussex',

    # Thames Valley
    'Bracknell Forest': 'Thames Valley', 'Buckinghamshire': 'Thames Valley', 'Cherwell': 'Thames Valley',
    'Milton Keynes': 'Thames Valley', 'Oxford': 'Thames Valley', 'Reading': 'Thames Valley',
    'Slough': 'Thames Valley', 'South Oxfordshire': 'Thames Valley', 'Vale of White Horse': 'Thames Valley',
    'West Berkshire': 'Thames Valley', 'West Oxfordshire': 'Thames Valley',
    'Windsor and Maidenhead': 'Thames Valley', 'Wokingham': 'Thames Valley'
}


iod2025 = pd.read_csv("File_7_IoD2025_All_Ranks_Scores_Deciles_Population_Denominators.csv")
# Mapping the LDOA names to the force mapping above where applicable
iod2025['Force'] = iod2025['Local Authority District name (2024)'].map(force_lookup)

# Showing how many records are not in covered by the 4 forces
excludedEntries = iod2025['Force'].isna().sum()
print(f"There are {excludedEntries} out of {len(iod2025)} LSOAs that are outside the 4 forces")

Feature Engineering

In [ ]:
# Adding column to flag if a neighbourhood is part of the top 10% of deprived areas
iodFiltered = iod2025.dropna(subset=['Force']).copy()
iodFiltered['is_decile_1'] = iodFiltered['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOAs)'] == 1

print(iodFiltered['is_decile_1'].value_counts())

Aggregation (Index of Deprivation Data)

In [ ]:
imd_final = iodFiltered.groupby('Force').agg(
    # Average Index of Deprivation Score per force
    AvgIMDScore=('Index of Multiple Deprivation (IMD) Score', 'mean'),
    # Average Income decile score per force
    AvgIncomeDecile=('Income Decile (where 1 is most deprived 10% of LSOAs)', 'mean'),
    # Number of neighbourhoods that force spans
    NeighbourhoodCount=('LSOA code (2021)', 'count'),
    # Number of neighbours in the force that are in the top 10 percent of deprivation
    HighlyDeprivedCount=('is_decile_1', 'sum')
    
).reset_index()

# Percentage of neighbourhoods in the force trhat are in the top 10 percent of deprivation
imd_final['HighlyDeprivedPercentage'] = (imd_final['HighlyDeprivedCount'] / imd_final['NeighbourhoodCount']) * 100

display(imd_final)

Final merging of all the data

In [ ]:
finalData = pd.merge(crimeAggregation, forceRatesBaseline, on=['Force', 'Year'], how='left')
completeData = pd.merge(finalData, imd_final, on=['Force'], how='left')

display(completeData)

Final Validation Checks

In [ ]:
# Check the number of duplicates in the final data
grainDuplicates = completeData.duplicated(subset=['Force', 'Year', 'LSOA name', 'Crime type']).sum()
print(f"Validation - Duplicate records at reporting grain: {grainDuplicates}")

# Check for number of null values in the data
print("\nFinal Missing Value Summary:")
print(completeData[['Force', 'Year', 'LSOA name', 'Crime type']].isnull().sum())

# General summary of final dataset statistics
print(f"\n--- Final Dataset Summary Statistics ---")
print(f"Total Grain Rows: {len(completeData)}")
print(f"Unique Police Forces Covered: {completeData['Force'].nunique()} ({completeData['Force'].unique()})")
print(f"Temporal Window Span: {completeData['Year'].min()} to {completeData['Year'].max()}")







Dataset Exporting

In [ ]:
completeData.to_csv('processedAndAggregatedCrimeData.csv', index=False)
print("\nSuccess: BI-ready reporting dataset successfully exported.")